# 04 · Paper tables and figures from all results

Attach the `dmmst-code` dataset and the outputs of notebooks 01–03 as inputs (Kaggle:
*Add Input → Your Work*), or run
locally after unzipping them into `results/new/`. Produces one combined report per
notebook plus everything in `report_all/`.

In [ ]:
INCLUDE_SMOKE = False   # True only to test this notebook on smoke-test outputs
GIT_URL = "https://github.com/Parsagh05/dmmst.git"

import sys
from pathlib import Path
from IPython.display import Markdown, Image, display

ON_KAGGLE = Path("/kaggle/working").is_dir()
REPO = next((f.parent.parent for f in (Path("/kaggle/input") if ON_KAGGLE else Path.cwd().resolve().parents[1]).rglob("runner.py")
             if f.parent.name == "scripts" and (f.parent.parent / "sat").is_dir()), None)
if REPO is None:  # standalone on Kaggle: clone the code
    import subprocess
    REPO = Path("/kaggle/working/dmmst")
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from scripts.report import build

SEARCH = [Path("/kaggle/input")] if ON_KAGGLE else [REPO / "results" / "new"]
OUT = (Path("/kaggle/working") if ON_KAGGLE else REPO / "results" / "new") / "report_all"
dirs = sorted({p.parent for root in SEARCH for p in root.rglob("*.meta.json")
               if "report" not in p.parts and (INCLUDE_SMOKE or not p.parent.name.endswith("_smoke"))})
print("\n".join(map(str, dirs)))

In [ ]:
for d in dirs:
    print("=" * 80, "\n", d.name)
    build([d], OUT / d.name)
    display(Markdown((OUT / d.name / "REPORT.md").read_text(encoding="utf-8")))
    for f in sorted((OUT / d.name).glob("sweep_*.png")):
        display(Image(filename=str(f)))